# Data Analysis of the filtered Measurements dataset

---

In [1]:
import duckdb

In [2]:
filtered_measurements_file = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered\filtered_measurements_encoded.parquet"
con = duckdb.connect()

In [3]:
book_state_counts = con.execute(f"""
    SELECT book_state, COUNT(*) AS count
    FROM '{filtered_measurements_file}'
    GROUP BY book_state
    ORDER BY count DESC
""").fetchdf()

print(book_state_counts)

   book_state     count
0           0  43712778
1           1    215341


In [5]:
# --- NULL % for all columns ---
print("Null percentage per column:")

# Get list of all column names using DESCRIBE
columns = con.execute(f"DESCRIBE SELECT * FROM '{filtered_measurements_file}'").fetchdf()['column_name'].tolist()

results = []
total_rows = con.execute(f"SELECT COUNT(*) FROM '{filtered_measurements_file}'").fetchone()[0]

for col in columns:
    null_count = con.execute(f"SELECT COUNT(*) - COUNT({col}) FROM '{filtered_measurements_file}'").fetchone()[0]
    null_percentage = round(100.0 * null_count / total_rows, 2)
    results.append((col, null_percentage))

# Sort and print
results.sort(key=lambda x: x[1], reverse=True)
for col, perc in results:
    print(f"{col}: {perc:.2f}% nulls")

Null percentage per column:
measure_step_number: 0.00% nulls
measure_value: 0.00% nulls
created_at: 0.00% nulls
booking_id: 0.00% nulls
book_state: 0.00% nulls
serial_number_id: 0.00% nulls
station_id: 0.00% nulls
measurement_name_encoded: 0.00% nulls
measurement_unit_encoded: 0.00% nulls
is_within_limits: 0.00% nulls


In [7]:
# --- Cardinality and sparsity for measurement_name_encoded & measurement_unit_encoded ---
columns_to_check = ['measurement_name_encoded', 'measurement_unit_encoded']

for col in columns_to_check:
    result = con.execute(f"""
        SELECT
            COUNT(DISTINCT {col}) AS unique_values,
            ROUND(100.0 * SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS missing_percentage
        FROM '{filtered_measurements_file}';
    """).fetchdf()
    print(f"\n=== 📌 {col} ===")
    print(result)

    # Top 10 frequent encoded values
    top_vals = con.execute(f"""
        SELECT {col}, COUNT(*) AS freq
        FROM '{filtered_measurements_file}'
        GROUP BY {col}
        ORDER BY freq DESC
        LIMIT 10;
    """).fetchdf()
    print(f"\nTop 10 frequent values for {col}:")
    print(top_vals)

con.close()


=== 📌 measurement_name_encoded ===
   unique_values  missing_percentage
0             23                 0.0

Top 10 frequent values for measurement_name_encoded:
   measurement_name_encoded      freq
0                     41832  19451859
1                     41833   9203260
2                     39144   6145608
3                     81352   2765968
4                     39143   2622580
5                      2689    634604
6                     39163    626608
7                     41607    582498
8                     41861    502332
9                     41503    415030

=== 📌 measurement_unit_encoded ===
   unique_values  missing_percentage
0             22                 0.0

Top 10 frequent values for measurement_unit_encoded:
   measurement_unit_encoded      freq
0                  13027095  13027095
1                   5273520   5273520
2                   5022683   5022683
3                   4531289   4531289
4                   3775669   3775669
5                   258133

In [9]:
# Connect
con = duckdb.connect()

# === Print schema (column names + types) ===
print("📋 Schema of the dataset:")
schema = con.execute(f"DESCRIBE SELECT * FROM '{filtered_measurements_file}'").fetchdf()
print(schema)

# === Row count ===
row_count = con.execute(f"SELECT COUNT(*) FROM '{filtered_measurements_file}'").fetchone()[0]
print(f"\n📊 Number of rows: {row_count}")

# === Summary statistics of numeric columns ===
print("\n📈 Summary statistics (numeric columns):")
summary = con.execute(f"""
SELECT
    COUNT(*) AS n_rows,
    MIN(measure_value) AS min_measure_value,
    MAX(measure_value) AS max_measure_value,
    AVG(measure_value) AS avg_measure_value,
    MIN(is_within_limits) AS min_within_limits,
    MAX(is_within_limits) AS max_within_limits,
    AVG(is_within_limits) AS avg_within_limits
FROM '{filtered_measurements_file}';
""").fetchdf()
print(summary)

# === Distinct value counts for encoded categorical columns (optional) ===
print("\n🔎 Distinct count of measurement_name_encoded and measurement_unit_encoded:")
cat_cardinality = con.execute(f"""
SELECT
    COUNT(DISTINCT measurement_name_encoded) AS n_measurement_names,
    COUNT(DISTINCT measurement_unit_encoded) AS n_measurement_units
FROM '{filtered_measurements_file}';
""").fetchdf()
print(cat_cardinality)

con.close()

📋 Schema of the dataset:
                column_name               column_type null   key default extra
0       measure_step_number                   INTEGER  YES  None    None  None
1             measure_value                    DOUBLE  YES  None    None  None
2                created_at  TIMESTAMP WITH TIME ZONE  YES  None    None  None
3                booking_id                   VARCHAR  YES  None    None  None
4                book_state                   INTEGER  YES  None    None  None
5          serial_number_id                   VARCHAR  YES  None    None  None
6                station_id                   VARCHAR  YES  None    None  None
7  measurement_name_encoded                    BIGINT  YES  None    None  None
8  measurement_unit_encoded                    BIGINT  YES  None    None  None
9          is_within_limits                   INTEGER  YES  None    None  None

📊 Number of rows: 43928119

📈 Summary statistics (numeric columns):
     n_rows  min_measure_value  max_m

In [10]:
con = duckdb.connect()

# === Descriptive Statistics for Numeric Columns ===
print("📊 Full descriptive statistics for numeric columns:")
desc_stats = con.execute(f"""
SELECT
    COUNT(*) AS n_rows,
    MIN(measure_value) AS min_measure_value,
    MAX(measure_value) AS max_measure_value,
    AVG(measure_value) AS avg_measure_value,
    MEDIAN(measure_value) AS median_measure_value,
    STDDEV_POP(measure_value) AS stddev_measure_value,

    MIN(is_within_limits) AS min_within_limits,
    MAX(is_within_limits) AS max_within_limits,
    AVG(is_within_limits) AS avg_within_limits,
    STDDEV_POP(is_within_limits) AS stddev_within_limits,

    MIN(measurement_name_encoded) AS min_name_encoded,
    MAX(measurement_name_encoded) AS max_name_encoded,
    AVG(measurement_name_encoded) AS avg_name_encoded,

    MIN(measurement_unit_encoded) AS min_unit_encoded,
    MAX(measurement_unit_encoded) AS max_unit_encoded,
    AVG(measurement_unit_encoded) AS avg_unit_encoded
FROM '{filtered_measurements_file}';
""").fetchdf()
print(desc_stats)

# === Full Pairwise Correlations (all numeric columns with each other) ===
numeric_cols = [
    'measure_value', 'measurement_name_encoded',
    'measurement_unit_encoded', 'is_within_limits'
]

print("\n📈 Pairwise Pearson correlations:")
for i, col1 in enumerate(numeric_cols):
    for col2 in numeric_cols[i + 1:]:
        corr = con.execute(f"""
            SELECT corr({col1}, {col2}) AS correlation
            FROM '{filtered_measurements_file}';
        """).fetchone()[0]
        print(f"Correlation({col1}, {col2}) = {corr:.4f}")

con.close()

📊 Full descriptive statistics for numeric columns:
     n_rows  min_measure_value  max_measure_value  avg_measure_value  \
0  43928119                0.0           100000.0          253.05308   

   median_measure_value  stddev_measure_value  min_within_limits  \
0                  50.0            374.814944                  0   

   max_within_limits  avg_within_limits  stddev_within_limits  \
0                  1           0.987969              0.109023   

   min_name_encoded  max_name_encoded  avg_name_encoded  min_unit_encoded  \
0                 2             81352      43236.466187             41832   

   max_unit_encoded  avg_unit_encoded  
0          13027095      6.322279e+06  

📈 Pairwise Pearson correlations:
Correlation(measure_value, measurement_name_encoded) = -0.0716
Correlation(measure_value, measurement_unit_encoded) = 0.5256
Correlation(measure_value, is_within_limits) = 0.0546
Correlation(measurement_name_encoded, measurement_unit_encoded) = -0.2597
Correlation(me